# Delivery Duration Prediction

## 1. Data preparation

- Load the dataset with pandas
- Inspect columns, data types, and unique values
- Clean strings, parse dates/times, and filter out invalid coordinates/distances

In [72]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [73]:
df = pd.read_csv('data/raw/train.csv')
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [74]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['id', 'delivery_person_id', 'delivery_person_age',
       'delivery_person_ratings', 'restaurant_latitude',
       'restaurant_longitude', 'delivery_location_latitude',
       'delivery_location_longitude', 'order_date', 'time_orderd',
       'time_order_picked', 'weatherconditions', 'road_traffic_density',
       'vehicle_condition', 'type_of_order', 'type_of_vehicle',
       'multiple_deliveries', 'festival', 'city', 'time_taken(min)'],
      dtype='object')

In [75]:
for col in df.columns:
	print(col)
	print(df[col].unique()[:5])
	print(df[col].nunique())
	print()

id
['0x4607 ' '0xb379 ' '0x5d6d ' '0x7a6a ' '0x70a2 ']
45593

delivery_person_id
['INDORES13DEL02 ' 'BANGRES18DEL02 ' 'BANGRES19DEL01 ' 'COIMBRES13DEL02 '
 'CHENRES12DEL01 ']
1320

delivery_person_age
['37' '34' '23' '38' '32']
23

delivery_person_ratings
['4.9' '4.5' '4.4' '4.7' '4.6']
29

restaurant_latitude
[22.745049 12.913041 12.914264 11.003669 12.972793]
657

restaurant_longitude
[75.892471 77.683237 77.6784   76.976494 80.249982]
518

delivery_location_latitude
[22.765049 13.043041 12.924264 11.053669 13.012793]
4373

delivery_location_longitude
[75.912471 77.813237 77.6884   77.026494 80.289982]
4373

order_date
['19-03-2022' '25-03-2022' '05-04-2022' '26-03-2022' '11-03-2022']
44

time_orderd
['11:30:00' '19:45:00' '08:30:00' '18:00:00' '13:30:00']
177

time_order_picked
['11:45:00' '19:50:00' '08:45:00' '18:10:00' '13:45:00']
193

weatherconditions
['conditions Sunny' 'conditions Stormy' 'conditions Sandstorms'
 'conditions Cloudy' 'conditions Fog']
7

road_traffic_density
[

In [76]:
df.dtypes

id                              object
delivery_person_id              object
delivery_person_age             object
delivery_person_ratings         object
restaurant_latitude            float64
restaurant_longitude           float64
delivery_location_latitude     float64
delivery_location_longitude    float64
order_date                      object
time_orderd                     object
time_order_picked               object
weatherconditions               object
road_traffic_density            object
vehicle_condition                int64
type_of_order                   object
type_of_vehicle                 object
multiple_deliveries             object
festival                        object
city                            object
time_taken(min)                 object
dtype: object

In [77]:
df['id'] = df['id'].astype(str).str.strip()
df = df.drop_duplicates()

df['time_taken(min)'] = df['time_taken(min)'].str.replace('(min)', '')
df['duration'] = pd.to_numeric(df['time_taken(min)'], errors='coerce')

categorical_map = {'weatherconditions': 'weather', 'road_traffic_density': 'traffic',
                   'type_of_vehicle': 'vehicle', 'type_of_order': 'order_type'}

for src, dest in categorical_map.items():
	df[dest] = (df[src].astype(str).str.strip().str.lower().str.removeprefix('conditions').str.strip().str.replace(' ', '_'))
	df[dest] = df[dest].replace({'': 'unknown', 'nan': 'unknown', 'none': 'unknown'})

df[['id', 'weather', 'traffic', 'vehicle', 'order_type', 'duration']].head()

,id,weather,traffic,vehicle,order_type,duration
0,0x4607,sunny,high,motorcycle,snack,24
1,0xb379,stormy,jam,scooter,snack,33
2,0x5d6d,sandstorms,low,motorcycle,drinks,26
3,0x7a6a,sunny,medium,motorcycle,buffet,21
4,0x70a2,cloudy,high,scooter,snack,30


In [78]:
for col in ["weather", "traffic", "vehicle", "order_type"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))
    print("Unknown percentage:",
          f"{df[col].eq('unknown').mean() * 100:.2f}%")


weather
weather
fog           7654
stormy        7586
cloudy        7536
sandstorms    7495
windy         7422
sunny         7284
unknown        616
Name: count, dtype: int64
Unknown percentage: 1.35%

traffic
traffic
low        15477
jam        14143
medium     10947
high        4425
unknown      601
Name: count, dtype: int64
Unknown percentage: 1.32%

vehicle
vehicle
motorcycle          26435
scooter             15276
electric_scooter     3814
bicycle                68
Name: count, dtype: int64
Unknown percentage: 0.00%

order_type
order_type
snack     11533
meal      11458
drinks    11322
buffet    11280
Name: count, dtype: int64
Unknown percentage: 0.00%


In [79]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')
times = pd.to_datetime(df['time_orderd'].astype(str).str.strip(), format='%H:%M:%S', errors='coerce')

df['order_hour'] = times.dt.hour.fillna(-1).astype(int)
df['hour_missing'] = (df['order_hour'] == -1).astype(int)

df[['order_date', 'order_hour', 'hour_missing']].head()

,order_date,order_hour,hour_missing
0,2022-03-19,11,0
1,2022-03-25,19,0
2,2022-03-19,8,0
3,2022-04-05,18,0
4,2022-03-26,13,0


In [80]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
	d_lat = math.radians(lat2 - lat1)
	d_lon = math.radians(lon2 - lon1)

	lat1 = math.radians(lat1)
	lat2 = math.radians(lat2)

	a = (math.sin(d_lat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2) ** 2)

	earth_radius = 6371.0088

	c = 2 * math.asin(math.sqrt(min(1.0, max(0.0, a))))

	return earth_radius * c

df['distance_km'] = [
	haversine_km(r_lat, r_lon, d_lat, d_lon)
  for r_lat, r_lon, d_lat, d_lon in zip(df['restaurant_latitude'], df['restaurant_longitude'],
    df['delivery_location_latitude'], df['delivery_location_longitude'])
]

df[['distance_km', 'duration']].head

<bound method NDFrame.head of        distance_km  duration
0         3.025153        24
1        20.183558        33
2         1.552760        26
3         7.790412        21
4         6.210147        30
...            ...       ...
45588     1.489848        32
45589    11.007750        36
45590     4.657202        16
45591     6.232402        26
45592    12.074412        36

[45593 rows x 2 columns]>

In [81]:
import json
from urllib.request import Request, urlopen

url = "https://nominatim.openstreetmap.org/search?country=India&format=json"
req = Request(url, headers={"User-Agent": "delivery-eta-project/1.0"})

with urlopen(req, timeout=10) as response:
    bbox = json.loads(response.read().decode())[0]["boundingbox"]
    
    print("OpenStreetMap India Bounding Box:")
    print(f"  Latitude : {float(bbox[0]):.2f} to {float(bbox[1]):.2f}  --> Project rule: [6, 36]")
    print(f"  Longitude: {float(bbox[2]):.2f} to {float(bbox[3]):.2f} --> Project rule: [67, 98]\n")

print("Raw CSV coordinate ranges (before cleaning):")
display(df[['restaurant_latitude', 'restaurant_longitude', 
            'delivery_location_latitude', 'delivery_location_longitude']].describe().loc[['min', 'max']])

OpenStreetMap India Bounding Box:
  Latitude : 6.55 to 35.67  --> Project rule: [6, 36]
  Longitude: 67.92 to 97.40 --> Project rule: [67, 98]

Raw CSV coordinate ranges (before cleaning):


,restaurant_latitude,restaurant_longitude,delivery_location_latitude,delivery_location_longitude
min,-30.905562,-88.366217,0.010000,0.010000
max,30.914057,88.433452,31.054057,88.563452


In [82]:
df.loc[(df['restaurant_latitude'] < 0) | (df['restaurant_longitude'] < 0),
  ['restaurant_latitude', 'restaurant_longitude', 'delivery_location_latitude', 'delivery_location_longitude']
].head(10)

,restaurant_latitude,restaurant_longitude,delivery_location_latitude,delivery_location_longitude
92,-27.163303,78.057044,27.233303,78.127044
283,-27.165108,78.015053,27.225108,78.075053
289,-18.551440,-73.804855,18.611440,73.864855
425,-30.366322,-78.070453,30.496322,78.200453
534,-12.325461,-76.632278,12.385461,76.692278
583,-12.316967,76.603067,12.326967,76.613067
762,-23.359194,-85.325447,23.449194,85.415447
1059,-26.477750,-80.351569,26.487750,80.361569
1091,-15.546594,73.760431,15.606594,73.820431
1157,-12.337978,76.616792,12.387978,76.666792


In [83]:
df['restaurant_latitude'].abs().describe()
df['restaurant_longitude'].abs().describe()

count    45593.000000
mean        70.782088
std         21.118611
min          0.000000
25%         73.170283
50%         75.898497
75%         78.046106
max         88.433452
Name: restaurant_longitude, dtype: float64

In [84]:
df['restaurant_latitude'] = df['restaurant_latitude'].abs()
df['restaurant_longitude'] = df['restaurant_longitude'].abs()

In [85]:
valid_coords = (
    df['restaurant_latitude'].between(6, 36) & df['restaurant_longitude'].between(67, 98) &
    df['delivery_location_latitude'].between(6, 36) & df['delivery_location_longitude'].between(67, 98)
)
valid_distance = (df['distance_km'] > 0) & (df['distance_km'] <= 100)
valid_target = df['duration'].notna() & (df['duration'] > 0)
valid_date = df['order_date'].notna()

df = df[valid_coords & valid_distance & valid_target & valid_date].copy()
df = df.sort_values(['order_date', 'id']).reset_index(drop=True)

print("Clean rows:", len(df))

Clean rows: 41522


In [86]:
df['weekend'] = (df['order_date'].dt.weekday >= 5).astype(int)
df['lunch'] = ((df['order_hour'] >= 11) & (df['order_hour'] < 14)).astype(int)
df['dinner'] = ((df['order_hour'] >= 17) & (df['order_hour'] < 21)).astype(int)

numerical = ['distance_km', 'order_hour', 'hour_missing', 'weekend', 'lunch', 'dinner']
categorical = ['weather', 'traffic', 'vehicle', 'order_type']

df[numerical + categorical + ['duration']].head()

,distance_km,order_hour,hour_missing,weekend,lunch,dinner,weather,traffic,vehicle,order_type,duration
0,1.468903,-1,1,0,0,0,unknown,unknown,scooter,drinks,19
1,2.984672,-1,1,0,0,0,unknown,unknown,electric_scooter,snack,31
2,3.026936,-1,1,0,0,0,unknown,unknown,motorcycle,meal,21
3,3.121293,-1,1,0,0,0,unknown,unknown,scooter,snack,30
4,6.176340,-1,1,0,0,0,fog,high,scooter,buffet,24


## 2. Setting up the validation framework
- Split 60% / 20% / 20% chronologically by unique `order_date`

In [88]:
days = np.sort(df['order_date'].unique())
n_days = len(days)

first = int(n_days * 0.6)
second = int(n_days * 0.8)

df_train = df[df['order_date'] < days[first]].copy()
df_val = df[(df['order_date'] >= days[first]) & (df['order_date'] < days[second])].copy()
df_test = df[df['order_date'] >= days[second]].copy()

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

len(df_train), len(df_val), len(df_test)


(23724, 9002, 8796)

In [90]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.duration.values
y_val = df_val.duration.values
y_test = df_test.duration.values
y_full_train = df_full_train.duration.values